# Agent-5: Interactive Choi Channel Widget

This notebook builds a local, IBM-free visualization widget for qubit quantum channels.  It follows the shared project convention

$$C_{\mathcal{E}} = \sum_{ij} |i\rangle\langle j| \otimes \mathcal{E}(|i\rangle\langle j|),$$

so the input system is the first tensor factor and trace preservation is checked by $\mathrm{Tr}_B(C_{\mathcal{E}})=I_A$.

**Palette:** blue `#2563eb`, green `#16a34a`, orange `#f97316`, red `#dc2626`, purple `#7c3aed`, gray `#475569`.

In [ ]:
import numpy as np

np.random.seed(42)

from widget_core import (
    CHANNEL_TYPES,
    build_widget,
    compute_indicators,
    format_indicator_text,
    get_channel_choi,
    render_dashboard_figure,
)

CHANNEL_TYPES

## 1. Design Goals

The widget is meant for fast educational feedback: a user changes a channel parameter and immediately sees the Choi matrix, the induced Bloch-sphere deformation, the Choi eigenspectrum, Kraus operators recovered from the Choi eigenvectors, and scalar CP/TP/fidelity indicators.

It deliberately runs with only NumPy, Matplotlib, and ipywidgets.  No IBM Quantum account, backend access, or Qiskit installation is required.

## 2. Architecture

- `channel_utils.py` contains local channel constructors and Choi utilities.
- `widget_core.py` contains all rendering, dashboard, and indicator logic.
- The dashboard uses one synchronized set of controls and redraws a four-panel Matplotlib figure on every slider or dropdown change.
- The supported channels are depolarizing, amplitude damping, phase damping, bit flip, phase flip, general Pauli, unital Bloch-axis scaling, and convex mixing of two supported channels.

## 3. Implementation: ipywidgets Dashboard

Run the next cell in JupyterLab or classic Notebook.  In static renderers it will still construct the widget object, but full slider interaction requires a live Jupyter frontend.

In [ ]:
dashboard = build_widget()
dashboard

## 4. Static Preview for Non-Interactive Runs

The same rendering code can produce a static dashboard figure, which is useful for README previews, notebook exports, and environments where widgets are disabled.

In [ ]:
choi = get_channel_choi("Amplitude damping", {"gamma": 0.35})
fig = render_dashboard_figure(choi, "Amplitude damping, gamma=0.35")
fig.savefig("figures/widget_preview.png", dpi=160)
fig

## 5. Indicator Sanity Checks

These lightweight checks show that the local channel utilities produce CP/TP matrices for standard physical channels.  The unital mode can intentionally enter non-CP regions when the Bloch scaling factors leave the CP tetrahedron; this is useful for teaching why Choi positivity matters.

In [ ]:
examples = {
    "identity": get_channel_choi("Identity", {}),
    "depolarizing p=0.2": get_channel_choi("Depolarizing", {"p": 0.2}),
    "amplitude damping gamma=0.35": get_channel_choi("Amplitude damping", {"gamma": 0.35}),
    "phase damping gamma=0.5": get_channel_choi("Phase damping", {"gamma": 0.5}),
    "pauli px=0.08 py=0.04 pz=0.12": get_channel_choi("Pauli", {"p_x": 0.08, "p_y": 0.04, "p_z": 0.12}),
}

for name, C in examples.items():
    print(name)
    print(format_indicator_text(compute_indicators(C)))
    print()

## 6. Educational Notes

Good quick experiments:

- Increase depolarizing `p` and watch the Bloch sphere contract uniformly while the Choi rank rises.
- Increase amplitude damping `gamma` and watch the Bloch ellipsoid shift toward `|0>` instead of remaining centered.
- Move the unital `lambda` sliders outside the CP tetrahedron and observe the CP indicator fail even though the TP indicator remains true.
- Use `Mix two channels` to see that convex mixtures of CP/TP channels remain CP/TP.